# Notebook de definición del problema y entorno reproducible — Fase 1

**Proyecto:** Predicción de accidente cerebrovascular (*stroke*)

---

### ¿Qué produce la Fase 1?

La Fase 2 recibe un conjunto de datos y lo deja listo para modelar. Para que eso sea
posible, alguien tuvo que decidir antes **qué problema se resuelve, con qué datos y sobre
qué entorno**. Ese es el trabajo de la Fase 1, y el flujo de este cuaderno es:

**Definir → Configurar → Verificar → Estructurar → Documentar → Versionar**

Cada paso se implementa como una **función** reutilizable y termina en una **comprobación
con evidencia**. La diferencia con la Fase 2 es el objeto de trabajo: allá se transforman
datos, aquí se construye y se verifica el andamiaje que hace que esa transformación sea
reproducible por otra persona.

> **Lo que esta fase NO hace.** No limpia, no imputa, no codifica ni escala. Si en este
> cuaderno aparece una imputación, el trabajo está adelantado de fase y la rúbrica de la
> Fase 2 se queda sin evidencia propia.

## Introducción

Según la Organización Mundial de la Salud, el accidente cerebrovascular es la segunda
causa de muerte a nivel mundial. El conjunto de datos escogido como caso de trabajo
permite estimar la probabilidad de que un paciente lo sufra a partir de variables
demográficas, clínicas y de hábitos.

**Objetivo general (Fase 1).** Definir la problemática y los objetivos del proyecto, y
dejar operativo un entorno de trabajo reproducible y versionado que sostenga las fases
siguientes.

**Objetivos específicos.**
- Declarar el problema, las preguntas que se responderán y los criterios de éxito.
- Verificar que el entorno de ejecución es el del proyecto y no el del sistema.
- Construir la estructura del repositorio y los artefactos que la hacen reproducible.
- Documentar la procedencia del conjunto de datos y evaluarla contra los criterios del curso.
- Dejar trazabilidad entre el mapa conceptual, el repositorio y este cuaderno.

**Entregables de la fase:** informe técnico, repositorio con estructura y *commits*
documentados, `README` técnico, este cuaderno ejecutado y el mapa conceptual de la
Formativa 1.

### Herramientas del ecosistema científico

| Pieza | Para qué la usamos | Qué falla si se omite |
|---|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones de las librerías del proyecto. | Un paquete «desaparece» o el compañero no puede reproducir. |
| JupyterLab + `ipykernel` | Ejecutar el cuaderno con el intérprete del proyecto. | `ModuleNotFoundError` con un paquete que sí está instalado. |
| `requirements.txt` | Declarar las dependencias exactas. | «A mí me funciona» y a nadie más. |
| Git | Registrar la historia del trabajo y su autoría. | No hay evidencia de contribución individual. |
| GitHub | Colaborar y publicar la evidencia evaluada. | El repositorio se convierte en un depósito de archivos. |
| `pathlib` | Rutas relativas independientes del sistema operativo. | `C:/Users/...` funciona en un solo computador del mundo. |

En esta fase se usan sobre todo módulos de la biblioteca estándar: el cuaderno construye y
verifica el entorno, todavía no analiza datos. `numpy` y `pandas` aparecen únicamente para
comprobar que están disponibles y para escribir las tablas de documentación.

In [ ]:
import sys                       # intérprete en uso: es lo que se verifica más abajo
import json                      # exportación de metadatos legibles
import platform                  # sistema operativo, para dejarlo en la bitácora
import subprocess                # consultas a Git desde el cuaderno
import shutil                    # localizar ejecutables (git) sin suponer que existen
import importlib                 # importar el módulo que este cuaderno va a escribir
from pathlib import Path         # manejo de rutas independiente del sistema operativo
from datetime import date

import numpy as np               # presente para verificar el entorno del proyecto
import pandas as pd              # tablas de documentación de la fase

# Reproducibilidad: la misma semilla que usa el cuaderno de la Fase 2.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 ·", date.today().isoformat())

## 1. Definición del problema

**Qué hace este paso.** Deja el problema escrito como una **estructura de datos**, no como
un párrafo suelto. Todo lo que el informe declara —problemática, objetivos, alcance,
criterios de éxito— queda aquí en un diccionario del que después se derivan el `README`,
los metadatos y la tabla de vinculación con el mapa conceptual.

**Por qué así.** Un dato escrito una sola vez no puede contradecirse consigo mismo. Si el
título del proyecto cambia, cambia en un lugar y se propaga a todos los artefactos. Es el
mismo principio que en la Fase 2 lleva a encapsular cada paso en una función en vez de
copiar y pegar.

In [ ]:
PROYECTO = {
    "titulo": "Predicción de accidente cerebrovascular",
    "grupo": "Grupo X",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Nombre Apellido", "Nombre Apellido", "Nombre Apellido"],
    "problematica": (
        "El accidente cerebrovascular es la segunda causa de muerte a nivel mundial y una "
        "fracción relevante de los casos ocurre en pacientes con factores de riesgo "
        "registrados con anterioridad. El problema es de clasificación binaria: estimar, a "
        "partir de variables demográficas, clínicas y de hábitos, la probabilidad de que un "
        "paciente presente el evento."
    ),
    "objetivo_general": (
        "Construir un flujo reproducible de análisis que, a partir de datos clínicos "
        "tabulares, permita caracterizar los factores asociados al accidente cerebrovascular."
    ),
    "objetivos_especificos": [
        "Definir el problema y establecer el entorno reproducible del proyecto (F1).",
        "Construir el pipeline de obtención, limpieza y transformación de datos (F2).",
        "Implementar el núcleo algorítmico con programación estructurada y POO (F3).",
        "Comunicar los hallazgos mediante visualización y un informe técnico (F4).",
    ],
    "preguntas": [
        "¿Qué variables concentran los valores faltantes y qué implica imputarlas?",
        "¿Qué proporción de la muestra corresponde a la clase positiva?",
        "¿Qué transformaciones exige cada rol analítico antes de cualquier modelo?",
    ],
    "criterios_exito": [
        "El repositorio se clona y el cuaderno corre completo sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los cuatro integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": ["Definición del problema", "Entorno reproducible", "Selección del conjunto"],
        "excluye": ["Limpieza e imputación", "Modelado predictivo", "Despliegue"],
    },
    "limitaciones": [
        "Los datos son secundarios: no se controló su recolección ni su representatividad.",
        "La variable objetivo está desbalanceada, lo que condiciona la evaluación posterior.",
        "El conjunto no incluye variables temporales, por lo que no hay análisis de evolución.",
    ],
}

La función siguiente presenta el proyecto en pantalla. Recibe el diccionario como
parámetro en vez de leer la variable global: así puede reutilizarse con la configuración de
cualquier grupo y probarse de forma aislada.

In [ ]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y con un mensaje que dice QUE falta, no solo que algo falló.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

El alcance y las limitaciones se presentan aparte porque cumplen una función distinta:
delimitan lo que **no** se hará. Es lo que evita que el informe de la Fase 4 sea juzgado
por algo que nunca estuvo comprometido.

In [ ]:
print("Alcance de la Fase 1")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nLimitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

> **Sobre las limitaciones.** Declararlas antes de haber analizado nada es señal de
> comprensión del problema, no de debilidad del trabajo. La alternativa —descubrirlas al
> final y omitirlas— es la que la rúbrica penaliza.

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba, con evidencia y no por confianza, que el cuaderno se
está ejecutando con el intérprete del proyecto y que las librerías declaradas están
disponibles en él.

**Por qué es la primera comprobación del proyecto.** Los dos errores más frecuentes del
módulo —un paquete que «desaparece» y un `ModuleNotFoundError` con una librería que sí se
instaló— no son errores de código: son el entorno o el kernel equivocados. Verificarlo
cuesta tres segundos y ahorra horas de diagnóstico sobre el archivo equivocado.

In [ ]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "sklearn"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# se instala scikit-learn, pero se importa sklearn.
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. Intérprete que ejecuta ESTE cuaderno. Si la ruta no contiene .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde aquí,
    #    no desde donde está guardado el archivo .ipynb.
    reporte["carpeta_trabajo"] = str(Path.cwd())
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Librerías del proyecto, con su versión.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se interrumpe el cuaderno: se informa qué instalar y con qué nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO = verificar_entorno(DEPENDENCIAS)

**Cómo se lee este resultado.** Cada línea corresponde a uno de los acoplamientos que la
guía de Git y GitHub describe en su Parte II:

| Lo que ve | Qué significa | Qué hacer |
| --- | --- | --- |
| El intérprete no contiene `.venv` | El kernel no es el del proyecto | *Kernel → Change Kernel* |
| Una librería aparece como `[FALTA]` | Se instaló fuera del entorno | Active el entorno y reinstale |
| La carpeta de trabajo no es la raíz del proyecto | Las rutas relativas fallarán | Abra JupyterLab desde la raíz |

> **Recomendación.** Deje esta celda como primera celda de todos sus cuadernos, también en
> las fases siguientes. Responde de inmediato la pregunta «¿el problema es mi código o mi
> entorno?», que es la que más tiempo consume cuando no se responde a tiempo.

## 3. Estructura del repositorio y artefactos

**Qué hace este paso.** Crea las carpetas del proyecto y escribe desde el cuaderno los dos
archivos que hacen reproducible el entorno: `.gitignore` y `requirements.txt`.

**Por qué desde el cuaderno.** Porque así quedan derivados de la configuración declarada en
la sección 1 y no de lo que alguien recordó escribir a mano. Es la misma lógica de la Fase
2: el artefacto se genera, se verifica y queda trazado.

In [ ]:
RAIZ = Path(".")                        # raíz del proyecto = carpeta desde la que se ejecuta

# Rutas de datos COMUNES a todas las fases: son exactamente las que usa el
# cuaderno de la Fase 2, de modo que un mismo archivo sirva a los dos.
DIR_CRUDO = RAIZ / "data" / "raw"          # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # salida del pipeline de la Fase 2
DIR_DOCS = RAIZ / "docs"                   # diccionario, fichas y metadatos
DIR_SRC = RAIZ / "src"                     # módulos propios reutilizables

# Una carpeta por fase para cuadernos e informes.
DIR_FASES = [RAIZ / f"F{n}" for n in (1, 2, 3, 4)]

for carpeta in [DIR_CRUDO, DIR_PROCESADO, DIR_DOCS, DIR_SRC, *DIR_FASES]:
    # parents=True crea las intermedias; exist_ok=True evita el error si ya existen,
    # que es lo que permite volver a ejecutar el cuaderno completo sin romperlo.
    carpeta.mkdir(parents=True, exist_ok=True)

print("Estructura creada bajo:", RAIZ.resolve())
for ruta in sorted(p for p in RAIZ.rglob("*") if p.is_dir() and ".git" not in p.parts):
    print("  ", ruta.as_posix() + "/")

> **Decisión que el grupo debe tomar y documentar.** La guía de desarrollo propone una
> variante en que cada fase tiene su propia carpeta `data/`. Aquí los datos son comunes a
> todas las fases y solo los cuadernos e informes se separan por fase, que es lo que
> supone el cuaderno de la Fase 2 al escribir `data/raw`. Cualquiera de las dos opciones
> sirve; lo que no puede ocurrir es que cada cuaderno suponga una ruta distinta. Si
> cambian de opción, cámbienla en los dos cuadernos a la vez y déjenlo escrito en el
> `README`.

### 3.1 El archivo `.gitignore`

Quedan fuera del repositorio tres cosas: lo que se regenera (el entorno virtual, las
cachés), lo que es local de cada equipo (archivos del sistema operativo) y lo que es
secreto (credenciales). Se escribe **antes** del primer `git add .`: después ya es tarde,
porque el archivo quedó en el historial.

In [ ]:
CONTENIDO_GITIGNORE = """# Entorno virtual: se reconstruye con requirements.txt
.venv/

# Caché de Python
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# Windows
Thumbs.db
desktop.ini

# macOS
.DS_Store

# Variables de entorno y secretos
.env

# Datos pesados y modelos
*.ckpt
*.pth
"""

ARCHIVO_GITIGNORE = RAIZ / ".gitignore"
ARCHIVO_GITIGNORE.write_text(CONTENIDO_GITIGNORE, encoding="utf-8")

# Verificación de ida y vuelta: se relee lo escrito antes de darlo por hecho.
lineas = [l for l in ARCHIVO_GITIGNORE.read_text(encoding="utf-8").splitlines()
          if l and not l.startswith("#")]
assert ".venv/" in lineas, "El entorno virtual debe estar ignorado."
print(f"{ARCHIVO_GITIGNORE} escrito con {len(lineas)} reglas activas.")

> **Atención — los datos.** Este `.gitignore` **no** ignora `data/`. Si el conjunto es
> liviano, versionarlo facilita que cualquiera reproduzca el análisis y es lo preferible
> para este curso; si supera unas decenas de MB, agréguenlo aquí y documenten en el
> `README` de dónde se descarga y en qué carpeta va. GitHub rechaza archivos sobre 100 MB.

### 3.2 El archivo `requirements.txt`

Declarar las dependencias es lo que separa «funciona en mi computador» de un entorno
reproducible. Se escriben con la versión exacta observada en el entorno, no con la última
que exista en el momento de instalar.

In [ ]:
def escribir_requirements(ruta, versiones):
    """
    Escribe requirements.txt a partir de las versiones observadas en el entorno.

    Parametros
    ----------
    ruta : Path
        Archivo de salida.
    versiones : dict
        Nombre de importacion -> version detectada (o None si falta).

    Retorna
    -------
    list of str
        Lineas escritas, para verificarlas.

    Lanza
    -----
    ValueError
        Si ninguna dependencia pudo detectarse.
    """
    lineas = []
    for nombre, version in versiones.items():
        if version is None:
            continue  # una librería ausente no se declara: primero se instala
        paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
        lineas.append(f"{paquete}=={version}")

    if not lineas:
        raise ValueError("No se detectó ninguna dependencia instalada.")

    # Se agregan las herramientas del flujo de trabajo, que no se importan
    # desde el código pero sí forman parte del entorno reproducible.
    lineas += ["jupyterlab", "notebook", "ipykernel"]
    ruta.write_text("\n".join(sorted(lineas)) + "\n", encoding="utf-8")
    return sorted(lineas)


ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"
requisitos = escribir_requirements(ARCHIVO_REQUIREMENTS, ENTORNO["versiones"])
print(ARCHIVO_REQUIREMENTS, "\n")
print("\n".join(requisitos))

> **Atención.** Este archivo se generó a partir de las librerías **realmente presentes** en
> el entorno. `pip freeze > requirements.txt` es la alternativa habitual y también sirve,
> con una diferencia: incluye además las dependencias indirectas, lo que da un entorno más
> fiel pero más difícil de leer. Elijan una de las dos y documenten cuál.

## 4. Módulos: del cuaderno a `src/`

**Qué hace este paso.** Escribe un módulo `src/utilidades.py` y lo importa de vuelta.

**Por qué.** La Unidad 1 pide funciones, excepciones **y módulos**. Un cuaderno documenta y
comunica; un módulo es lo que permite que la misma función se use en las cuatro fases sin
copiarla. Aquí se mueve una función pequeña pero real: la que normaliza los nombres de las
columnas, que las fases siguientes usarán apenas carguen datos.

In [ ]:
CODIGO_MODULO = """\
\"\"\"Utilidades compartidas del proyecto MCDI500 - Grupo X.

Este modulo lo genera el cuaderno de la Fase 1 y lo importan las fases siguientes.
\"\"\"

import re


class ProyectoError(Exception):
    \"\"\"Error propio del proyecto: permite distinguirlo de los de las librerias.\"\"\"


def normalizar_nombre(texto):
    \"\"\"Convierte un nombre de columna a minusculas, sin espacios ni signos.

    Ejemplo
    -------
    >>> normalizar_nombre('Residence Type ')
    'residence_type'
    \"\"\"
    if not isinstance(texto, str):
        raise ProyectoError(f\"Se esperaba texto y se recibio {type(texto).__name__}.\")
    limpio = texto.strip().lower()
    limpio = re.sub(r\"[^a-z0-9]+\", \"_\", limpio)   # todo lo que no sea letra o digito -> _
    return limpio.strip(\"_\")


def normalizar_columnas(nombres):
    \"\"\"Aplica normalizar_nombre a una lista y verifica que no se produzcan duplicados.\"\"\"
    normalizados = [normalizar_nombre(n) for n in nombres]
    if len(set(normalizados)) != len(normalizados):
        raise ProyectoError(\"La normalizacion produjo nombres duplicados.\")
    return normalizados
"""

ARCHIVO_MODULO = DIR_SRC / "utilidades.py"
ARCHIVO_MODULO.write_text(CODIGO_MODULO, encoding="utf-8")
print("Módulo escrito en:", ARCHIVO_MODULO)

Para importarlo hay que agregar `src/` a la lista de rutas donde Python busca módulos.
`invalidate_caches()` es necesario porque el archivo se acaba de crear: sin esa llamada,
Python puede no ver un módulo escrito después de arrancar el intérprete.

In [ ]:
if str(DIR_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(DIR_SRC.resolve()))

importlib.invalidate_caches()
import utilidades                      # el módulo recién escrito
importlib.reload(utilidades)           # por si el cuaderno se vuelve a ejecutar

print("Importado:", utilidades.__name__, "desde", Path(utilidades.__file__).as_posix())
print(utilidades.normalizar_nombre("Residence Type "))
print(utilidades.normalizar_columnas(["Avg Glucose Level", "BMI", "smoking status"]))

### 4.1 Pruebas del módulo (casos normal, límite y excepción)

Las mismas tres categorías que exige la rúbrica en la Fase 2 se aplican aquí. `assert`
detiene el cuaderno si una condición no se cumple, dejando constancia de la falla en la
salida ejecutada.

In [ ]:
# --- Caso normal ---
assert utilidades.normalizar_nombre("Avg Glucose Level") == "avg_glucose_level"
assert utilidades.normalizar_columnas(["A B", "c-d"]) == ["a_b", "c_d"]
print("[OK] Caso normal: normalización correcta")

# --- Caso límite: signos consecutivos y espacios en los extremos ---
assert utilidades.normalizar_nombre("  ¿BMI (kg/m2)?  ") == "bmi_kg_m2"
assert utilidades.normalizar_columnas([]) == []
print("[OK] Caso límite: signos y lista vacía")

# --- Excepción 1: tipo incorrecto ---
try:
    utilidades.normalizar_nombre(42)
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (tipo): {e}")

# --- Excepción 2: la normalización colapsa dos nombres en uno ---
# "BMI" y " bmi " producen ambos "bmi": el módulo lo detecta en vez de perder una columna.
try:
    utilidades.normalizar_columnas(["BMI", " bmi "])
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (duplicados): {e}")

> **Por qué una excepción propia.** `ProyectoError` hereda de `Exception` y permite
> distinguir un fallo del proyecto de uno de pandas o de scikit-learn. Cuando el cuaderno
> de la Fase 3 capture errores, podrá tratar de forma distinta «los datos venían mal» y
> «mi función recibió lo que no esperaba».

## 5. Selección y ficha del conjunto de datos

**Qué hace este paso.** Documenta la procedencia del conjunto y la evalúa contra los
criterios de selección del curso.

**Por qué en la Fase 1.** Porque la elección del conjunto condiciona todo lo que viene
después. Un conjunto ya limpio convierte la sección de preprocesamiento de la Fase 2 en un
trámite y hace perder el criterio de mayor peso de la Sumativa 1.

### 5.1 Procedencia

| Campo | Valor |
| --- | --- |
| Título | Stroke Prediction Dataset |
| Autor | fedesoriano |
| Plataforma | Kaggle |
| Enlace | `https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset` |
| Estructura esperada | 5.110 filas × 12 columnas · 201 nulos en `bmi` |
| Unidad de observación | Un paciente |
| Licencia | Abierta, declarada en la plataforma |

**Referencia en APA 7 para el informe:**

> fedesoriano. (2021). *Stroke Prediction Dataset* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

Documentar la procedencia no es un trámite: sin ella nadie puede verificar sobre qué
versión del archivo se trabajó.

In [ ]:
FICHA = {
    "titulo": "Stroke Prediction Dataset",
    "autor": "fedesoriano",
    "plataforma": "Kaggle",
    "url": "https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset",
    "licencia_abierta": True,
    "unidad_observacion": "Un paciente",
    "filas": 5110,
    "columnas": 12,
    "tamano_mb": 0.3,
    # Conteo de variables por ROL ANALÍTICO, no por tipo de dato: es el rol
    # el que determina qué preprocesamiento habrá que justificar en la Fase 2.
    "roles": {
        "continua": 3,          # age, avg_glucose_level, bmi
        "discreta": 0,
        "binaria": 4,           # hypertension, heart_disease, ever_married, stroke
        "nominal": 3,           # gender, work_type, Residence_type
        "ordinal": 0,
        "fecha": 0,
        "alta_cardinalidad": 0,
        "identificador": 1,     # id
    },
    "pct_nulos_max_variable": 3.9,   # bmi
    "pct_nulos_min_no_cero": 3.9,
}

### 5.2 Evaluación contra los criterios del curso

La función compara la ficha con los mínimos definidos para el proyecto y devuelve una
tabla con el veredicto por criterio. No decide por el grupo: expone lo que falta.

In [ ]:
CRITERIOS = {
    "filas": ("Filas", 2000, "Permite agrupar por categoría sin grupos vacíos"),
    "columnas": ("Columnas", 12, "Asegura variedad de roles analíticos"),
    "continua": ("Numéricas continuas", 2, "Escalamiento y valores atípicos"),
    "discreta": ("Numérica discreta", 1, "Obliga a decidir número frente a categoría"),
    "nominal": ("Categóricas nominales", 2, "Codificación One-Hot"),
    "ordinal_o_binaria": ("Binaria u ordinal", 1, "Codificación con orden declarado"),
    "fecha": ("Fecha", 1, "Parseo y derivación de variables temporales"),
    "alta_cardinalidad": ("Texto o alta cardinalidad", 1, "Normalización y agrupación"),
}


def evaluar_criterios(ficha, criterios=CRITERIOS):
    """
    Compara una ficha de conjunto de datos con los minimos exigidos por el curso.

    Parametros
    ----------
    ficha : dict
        Ficha del conjunto, con las claves 'filas', 'columnas' y 'roles'.
    criterios : dict
        Criterio -> (etiqueta, minimo, motivo).

    Retorna
    -------
    pd.DataFrame
        Una fila por criterio, con el valor observado y el veredicto.

    Lanza
    -----
    KeyError
        Si la ficha no declara los roles analiticos.
    """
    if "roles" not in ficha:
        raise KeyError("La ficha debe declarar el conteo de variables por rol analítico.")

    roles = ficha["roles"]
    observado = {
        "filas": ficha.get("filas", 0),
        "columnas": ficha.get("columnas", 0),
        "continua": roles.get("continua", 0),
        "discreta": roles.get("discreta", 0),
        "nominal": roles.get("nominal", 0),
        # Binaria u ordinal cuenta como un solo criterio: basta con una de las dos.
        "ordinal_o_binaria": roles.get("ordinal", 0) + roles.get("binaria", 0),
        "fecha": roles.get("fecha", 0),
        "alta_cardinalidad": roles.get("alta_cardinalidad", 0),
    }

    filas = []
    for clave, (etiqueta, minimo, motivo) in criterios.items():
        valor = observado[clave]
        filas.append({
            "criterio": etiqueta,
            "mínimo": minimo,
            "observado": valor,
            "cumple": "sí" if valor >= minimo else "NO",
            "por qué se pide": motivo,
        })

    # Los faltantes son la condición más importante: un conjunto ya limpio deja
    # sin evidencia la sección de preprocesamiento de la Fase 2.
    pct = ficha.get("pct_nulos_min_no_cero", 0.0)
    filas.append({
        "criterio": "Alguna variable con ≥1% de nulos",
        "mínimo": 1.0, "observado": pct,
        "cumple": "sí" if pct >= 1.0 else "NO",
        "por qué se pide": "Un conjunto perfecto no tiene preprocesamiento que justificar",
    })
    filas.append({
        "criterio": "Ninguna variable sobre 60% de nulos",
        "mínimo": 60.0, "observado": ficha.get("pct_nulos_max_variable", 0.0),
        "cumple": "sí" if ficha.get("pct_nulos_max_variable", 0.0) <= 60.0 else "NO",
        "por qué se pide": "Por encima de eso la variable no es recuperable",
    })
    tabla = pd.DataFrame(filas)
    # Formato: los mínimos y los valores observados se muestran sin decimales
    # innecesarios; la comparación ya se hizo sobre los números.
    for col in ("mínimo", "observado"):
        tabla[col] = tabla[col].map(lambda v: f"{v:g}")
    return tabla


evaluacion = evaluar_criterios(FICHA)
evaluacion

In [ ]:
incumplidos = evaluacion.loc[evaluacion["cumple"] == "NO", "criterio"].tolist()
print(f"Criterios evaluados : {len(evaluacion)}")
print(f"Criterios cumplidos : {len(evaluacion) - len(incumplidos)}")
if incumplidos:
    print("\nNo cumple:")
    for c in incumplidos:
        print("  ·", c)

> **Léase con atención el resultado.** El conjunto de *stroke* **no cumple** los criterios
> del proyecto: no tiene variable de fecha, ni variable discreta, ni una de alta
> cardinalidad. Eso no es un descuido de la ficha: es exactamente la razón por la que se
> usa como **caso docente** y queda excluido como conjunto de proyecto. Su grupo debe
> elegir otro, y esta misma función es la que debe devolver «sí» en todas las filas antes
> de que lo declaren en el informe.

**Para su proyecto.** Copie la celda siguiente, complete la ficha de su conjunto candidato
y ejecútela. Mientras alguna fila diga `NO`, el conjunto no está aprobado.

In [ ]:
# TODO — Complete esta ficha con los datos de SU conjunto candidato y vuelva a evaluar.
FICHA_GRUPO = {
    "titulo": "TODO: título del conjunto",
    "autor": "TODO",
    "plataforma": "TODO",
    "url": "TODO",
    "licencia_abierta": False,
    "filas": 0,
    "columnas": 0,
    "roles": {"continua": 0, "discreta": 0, "binaria": 0, "nominal": 0,
              "ordinal": 0, "fecha": 0, "alta_cardinalidad": 0, "identificador": 0},
    "pct_nulos_max_variable": 0.0,
    "pct_nulos_min_no_cero": 0.0,
}

evaluar_criterios(FICHA_GRUPO)

### 5.3 Diccionario de variables

El diccionario declara el **rol analítico** de cada variable. Es la decisión que determina
todo el preprocesamiento de la Fase 2: dos columnas `int64` pueden ser un identificador,
una variable discreta o una binaria, y cada una exige un tratamiento distinto.

In [ ]:
DICCIONARIO = [
    ("id",                "identificador", "Identificador único del paciente"),
    ("gender",            "nominal",       "Female, Male, Other"),
    ("age",               "continua",      "Edad en años"),
    ("hypertension",      "binaria",       "1 = diagnóstico de hipertensión"),
    ("heart_disease",     "binaria",       "1 = enfermedad cardiaca"),
    ("ever_married",      "binaria",       "No / Yes"),
    ("work_type",         "nominal",       "Govt_job, Never_worked, Private, Self-employed, children"),
    ("Residence_type",    "nominal",       "Rural / Urban"),
    ("avg_glucose_level", "continua",      "Nivel medio de glucosa en sangre"),
    ("bmi",               "continua",      "Índice de masa corporal; contiene faltantes"),
    ("smoking_status",    "nominal",       "Unknown, formerly smoked, never smoked, smokes"),
    ("stroke",            "binaria",       "Variable objetivo: 1 = sufrió el evento"),
]

diccionario = pd.DataFrame(DICCIONARIO, columns=["variable", "rol_analitico", "descripcion"])

# Verificación: el diccionario debe cubrir exactamente las columnas declaradas en la ficha.
assert len(diccionario) == FICHA["columnas"], "El diccionario no coincide con la ficha."
print(diccionario["rol_analitico"].value_counts().to_string())
diccionario

> **`Unknown` no es un valor faltante.** En `smoking_status` representa cerca de un tercio
> de la muestra. Declararlo aquí como categoría propia, y no como nulo, evita que en la
> Fase 2 se impute un tercio del conjunto por inercia.

### 5.4 Comprobación del archivo (si ya está descargado)

Esta celda no descarga nada. Comprueba si el archivo está en `data/raw` y, en ese caso,
contrasta la estructura real con la declarada en la ficha. Si aún no lo descargan, informa
qué falta y el cuaderno sigue corriendo: la Fase 1 no depende del archivo.

In [ ]:
ARCHIVO = DIR_CRUDO / "healthcare-dataset-stroke-data.csv"

if ARCHIVO.exists():
    # na_values deja constancia de que los faltantes vienen como el texto 'N/A'.
    df_crudo = pd.read_csv(ARCHIVO, na_values=["N/A"])
    print("Archivo encontrado :", ARCHIVO.as_posix())
    print("Dimensiones reales :", df_crudo.shape)
    print("Declarado en ficha :", (FICHA["filas"], FICHA["columnas"]))
    coincide = df_crudo.shape == (FICHA["filas"], FICHA["columnas"])
    print("Coincidencia       :", "[OK]" if coincide else "[REVISAR] ficha y archivo difieren")
    print("\nNulos por columna (solo las que tienen):")
    print(df_crudo.isna().sum().loc[lambda s: s > 0].to_string())
else:
    df_crudo = None
    print("[PENDIENTE] El archivo aún no está en", DIR_CRUDO.as_posix())
    print("Descárguelo desde:", FICHA["url"])
    print("y guárdelo como  :", ARCHIVO.as_posix())
    print("\nLa Fase 1 no requiere el archivo: la exploración es trabajo de la Fase 2.")

## 6. Control de versiones

**Qué hace este paso.** Consulta el estado de Git desde el propio cuaderno: si está
instalado, si la identidad está configurada, si la carpeta es un repositorio y qué
*commits* tiene.

**Por qué se consulta y no se ejecuta.** Este cuaderno **no** hace `git init`, `add` ni
`commit`. Versionar es una decisión de quien trabaja, y automatizarla desde un cuaderno
produce historiales sin sentido. Lo que sí corresponde es dejar constancia verificable del
estado del repositorio en el momento de la entrega.

In [ ]:
def git(*argumentos):
    """
    Ejecuta un comando de Git y devuelve su salida como texto.

    Retorna
    -------
    (bool, str)
        Exito de la ejecucion y salida (o mensaje de error).
    """
    if shutil.which("git") is None:
        return False, "Git no está instalado o no está en el PATH."
    try:
        proceso = subprocess.run(["git", *argumentos], capture_output=True,
                                 text=True, timeout=20)
    except (OSError, subprocess.SubprocessError) as e:
        return False, f"No se pudo ejecutar Git: {e}"
    if proceso.returncode != 0:
        # stderr trae el motivo real; se devuelve tal cual para no ocultarlo.
        return False, proceso.stderr.strip() or "Git terminó con error."
    return True, proceso.stdout.strip()


ok, salida = git("--version")
print("Git                :", salida if ok else f"[AVISO] {salida}")

for clave in ("user.name", "user.email"):
    ok_cfg, valor = git("config", "--get", clave)
    estado = valor if ok_cfg and valor else "[FALTA] configúrelo con git config --global"
    print(f"{clave:19}:", estado)

> **Atención.** El correo configurado en Git debe ser **el mismo** de la cuenta de GitHub.
> Si no coinciden, los *commits* aparecen en el historial sin vincularse al perfil y en la
> revisión no es posible atribuir el trabajo a su autor. Corregirlo después implica
> reescribir el historial, que es un procedimiento delicado.

In [ ]:
es_repo, _ = git("rev-parse", "--is-inside-work-tree")

if es_repo:
    ok_rama, rama = git("rev-parse", "--abbrev-ref", "HEAD")
    print("Rama actual        :", rama if ok_rama else "sin commits todavía")

    ok_log, log = git("log", "--oneline", "-5")
    print("\nÚltimos commits")
    print(log if ok_log and log else "  (todavía no hay commits)")

    # --porcelain da una salida estable, pensada para ser leída por un programa.
    ok_st, estado = git("status", "--porcelain")
    pendientes = [l for l in estado.splitlines() if l.strip()] if ok_st else []
    print(f"\nArchivos con cambios sin registrar: {len(pendientes)}")
    for linea in pendientes[:10]:
        print("  ", linea)

    ok_rem, remoto = git("remote", "-v")
    print("\nRemoto             :", remoto.splitlines()[0] if ok_rem and remoto
          else "[FALTA] agregue el remoto con git remote add origin URL")
else:
    print("[PENDIENTE] Esta carpeta todavía no es un repositorio Git.")
    print("Desde la terminal, en la raíz del proyecto:")
    print("  git init")
    print("  git add .gitignore requirements.txt README.md")
    print('  git commit -m "docs: estructura inicial del proyecto"')

**Convención de *commits* del curso.** Los prefijos `docs`, `data`, `feat` y `fix`
ordenan el historial y permiten leerlo como una bitácora. Un *commit* por bloque de trabajo
con sentido propio, no uno al final del día.

| Prefijo | Cuándo | Ejemplo |
| --- | --- | --- |
| `docs` | Documentación, README, informe | `docs: agrega ficha del conjunto de datos` |
| `data` | Incorporación o actualización de datos | `data: agrega archivo original en data/raw` |
| `feat` | Código nuevo que aporta funcionalidad | `feat: agrega funcion de normalizacion de columnas` |
| `fix` | Corrección de un defecto | `fix: corrige ruta relativa en la carga` |

> **Atención.** Evite tildes y caracteres especiales en los mensajes de *commit*: según la
> configuración de cada terminal, aparecen corruptos en el historial compartido.

## 7. Validación técnica de la fase

Verificar significa demostrar, con evidencia, que el resultado es correcto. La función
`validar_fase1` reúne las comprobaciones de la fase y usa `assert`: si una condición no se
cumple, el cuaderno se detiene con un error, dejando constancia de la falla.

Se comprueba:
1. **Estructura** — que existan las carpetas comprometidas.
2. **Artefactos** — que `.gitignore` y `requirements.txt` estén escritos y no vacíos.
3. **Definición** — que la configuración del proyecto tenga sus claves obligatorias.
4. **Coherencia** — que el diccionario cubra exactamente las columnas de la ficha.
5. **Módulo** — que `src/utilidades.py` sea importable.

In [ ]:
def validar_fase1(raiz, config, ficha, diccionario):
    """
    Comprueba que la Fase 1 dejo lo que la rubrica exige.
    Lanza AssertionError si alguna comprobacion falla.
    """
    print("VALIDACION DE LA FASE 1")
    print("-" * 45)

    # 1. Estructura de carpetas
    esperadas = ["data/raw", "data/processed", "docs", "src", "F1", "F2", "F3", "F4"]
    faltan = [c for c in esperadas if not (raiz / c).is_dir()]
    assert not faltan, f"Faltan carpetas: {faltan}"
    print(f"[OK] Estructura completa ({len(esperadas)} carpetas)")

    # 2. Artefactos del entorno reproducible
    for nombre in (".gitignore", "requirements.txt"):
        archivo = raiz / nombre
        assert archivo.exists(), f"Falta {nombre}."
        assert archivo.stat().st_size > 0, f"{nombre} está vacío."
        print(f"[OK] {nombre} presente ({archivo.stat().st_size} bytes)")

    # 3. Definición del problema
    for clave in ("titulo", "problematica", "objetivo_general", "objetivos_especificos"):
        assert config.get(clave), f"La configuración no declara '{clave}'."
    assert len(config["objetivos_especificos"]) >= 3, "Se esperan al menos tres objetivos."
    print("[OK] Definición del problema completa")

    # 4. Coherencia entre ficha y diccionario
    assert len(diccionario) == ficha["columnas"], "Diccionario y ficha no coinciden."
    roles_declarados = set(diccionario["rol_analitico"])
    assert roles_declarados, "El diccionario no declara roles analíticos."
    print(f"[OK] Diccionario coherente ({len(diccionario)} variables, "
          f"{len(roles_declarados)} roles)")

    # 5. Módulo importable
    assert (raiz / "src" / "utilidades.py").exists(), "Falta el módulo src/utilidades.py."
    assert callable(getattr(utilidades, "normalizar_columnas", None)), \
        "El módulo no expone normalizar_columnas."
    print("[OK] Módulo src/utilidades.py importable")

    print(f"\n[INFO] Conjunto declarado: {ficha['titulo']}")
    print(f"[INFO] Archivo descargado: {'sí' if ARCHIVO.exists() else 'todavía no'}")
    return True


validar_fase1(RAIZ, PROYECTO, FICHA, diccionario)

## 8. Vinculación con el mapa conceptual

La rúbrica de la Sumativa 1 evalúa por separado que el mapa conceptual de la Formativa 1
se corresponda con lo efectivamente construido. La tabla siguiente hace explícita esa
correspondencia: cada nodo del mapa, dónde está implementado y qué evidencia lo respalda.

In [ ]:
VINCULACION = [
    # (nodo del mapa conceptual, dónde se implementa, evidencia verificable)
    ("Entorno virtual",        "Sección 2 · verificar_entorno", "Ruta del intérprete y versiones"),
    ("Gestión de dependencias", "Sección 3.2 · requirements.txt", "Archivo generado desde el entorno"),
    ("Estructura del proyecto", "Sección 3 · carpetas F1–F4",     "Árbol impreso y validado"),
    ("Módulos reutilizables",   "Sección 4 · src/utilidades.py",  "Módulo importado y probado"),
    ("Fuente de datos",         "Sección 5 · FICHA y DICCIONARIO", "Ficha, cita APA 7 y roles"),
    ("Control de versiones",    "Sección 6 · consultas a Git",     "Rama, commits y estado"),
    ("Documentación científica", "Sección 9 · README y metadatos", "Archivos en docs/"),
]

vinculacion = pd.DataFrame(VINCULACION,
                           columns=["nodo_mapa", "donde_se_implementa", "evidencia"])

# Ningún nodo del mapa puede quedar sin evidencia: eso es lo que la rúbrica revisa.
assert vinculacion["evidencia"].str.strip().ne("").all(), "Hay nodos sin evidencia."
print(f"{len(vinculacion)} nodos del mapa conceptual, todos con evidencia asociada.\n")
vinculacion

> **Cómo se escribe esto en el informe.** La tabla no basta por sí sola. Cada fila debe
> poder señalarse en el repositorio: un archivo, una celda o un *commit*. Un nodo del mapa
> que no tiene dónde apuntar es un nodo que se dibujó pero no se construyó.

## 9. Persistencia y trazabilidad

El resultado debe quedar guardado y documentado. Esta sección produce los artefactos que se
versionan en el repositorio y se citan en el informe técnico.

In [ ]:
def generar_readme(config, ficha, requisitos, ruta):
    """Escribe el README tecnico del repositorio a partir de la configuracion."""
    integrantes = "\n".join(f"- {n} (@usuario-github)" for n in config["integrantes"])
    dependencias = "\n".join(f"- {r}" for r in requisitos)

    contenido = f"""# {config['titulo']} — {config['grupo']}

{config['problematica']}

## Integrantes
{integrantes}

## Datos
- Fuente: {ficha['titulo']} ({ficha['autor']}, {ficha['plataforma']})
- Enlace: {ficha['url']}
- Estructura: {ficha['filas']} filas x {ficha['columnas']} columnas
- Ubicación esperada: `data/raw/`

## Estructura del repositorio
```
data/raw/        datos originales, sin modificar
data/processed/  datos tras limpieza y transformación (F2)
docs/            diccionario, fichas y metadatos
src/             módulos reutilizables del proyecto
F1/ F2/ F3/ F4/  cuadernos e informes de cada fase
```

## Requisitos y ejecución
Python 3.13

    python -m venv .venv
    source .venv/Scripts/activate    # Windows, Git Bash
    # source .venv/bin/activate      # macOS y Linux
    python -m pip install -r requirements.txt

Ejecutar los cuadernos en orden desde la raíz del proyecto.

### Dependencias declaradas
{dependencias}

## Convención de commits
Prefijos usados: docs, data, feat, fix.

## Decisiones técnicas
- Los datos son comunes a todas las fases (`data/`); los cuadernos se separan por fase.
- Semilla aleatoria fijada en {SEMILLA} para asegurar reproducibilidad.
- Registro breve de las decisiones relevantes y su motivo, actualizado al cerrar cada fase.
"""
    ruta.write_text(contenido, encoding="utf-8")
    return contenido


ARCHIVO_README = RAIZ / "README.md"
contenido = generar_readme(PROYECTO, FICHA, requisitos, ARCHIVO_README)
print(f"{ARCHIVO_README} escrito ({len(contenido.splitlines())} líneas)\n")
print("\n".join(contenido.splitlines()[:14]), "\n...")

In [ ]:
# Artefactos de documentación de la fase, en formato tabular para el informe.
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
evaluacion.to_csv(DIR_DOCS / "evaluacion_criterios_dataset.csv", index=False)
vinculacion.to_csv(DIR_DOCS / "vinculacion_mapa_conceptual.csv", index=False)

# Metadatos de la fase: lo que permite reconstruir en qué condiciones se ejecutó.
METADATOS = {
    "proyecto": PROYECTO["titulo"],
    "grupo": PROYECTO["grupo"],
    "fase": "F1",
    "fecha_ejecucion": date.today().isoformat(),
    "semilla": SEMILLA,
    "python": sys.version.split()[0],
    "sistema": f"{platform.system()} {platform.release()}",
    "interprete": ENTORNO["interprete"],
    "entorno_virtual": ENTORNO["entorno_virtual"],
    "dependencias": ENTORNO["versiones"],
    "dataset": {"titulo": FICHA["titulo"], "url": FICHA["url"],
                "archivo_descargado": ARCHIVO.exists()},
}
(DIR_DOCS / "metadatos_fase1.json").write_text(
    json.dumps(METADATOS, indent=2, ensure_ascii=False), encoding="utf-8")

for archivo in sorted(DIR_DOCS.iterdir()):
    print(f"  {archivo.as_posix():48} {archivo.stat().st_size:6} bytes")

In [ ]:
# Resumen de cierre: qué existía al empezar y qué queda al terminar la fase.
resumen = pd.DataFrame([
    {"artefacto": "Carpetas del proyecto",
     "estado": len([p for p in RAIZ.rglob("*") if p.is_dir() and ".git" not in p.parts])},
    {"artefacto": "Archivos en docs/", "estado": len(list(DIR_DOCS.iterdir()))},
    {"artefacto": "Dependencias declaradas", "estado": len(requisitos)},
    {"artefacto": "Variables en el diccionario", "estado": len(diccionario)},
    {"artefacto": "Nodos del mapa vinculados", "estado": len(vinculacion)},
    {"artefacto": "Criterios de dataset incumplidos", "estado": len(incumplidos)},
])
resumen

## Conclusiones y trazabilidad

La fase deja operativo un entorno verificado, una estructura de repositorio con sus
artefactos de reproducibilidad (`.gitignore`, `requirements.txt`, `README.md`), un módulo
propio importable y la documentación del conjunto de datos con su evaluación contra los
criterios del curso. Cada paso se implementó como una función documentada y verificada
(`presentar_proyecto`, `verificar_entorno`, `escribir_requirements`, `evaluar_criterios`,
`git`, `validar_fase1`, `generar_readme`).

**Trazabilidad con el repositorio (F1):** este cuaderno se ubica en la carpeta `F1/`; el
`README` documenta dependencias e instrucciones de ejecución; los archivos de `docs/`
sostienen las tablas del informe técnico, y el historial de *commits* refleja el avance
descrito en él.

---

## 10. Reflexión técnica

Este apartado es obligatorio en la entrega. Redáctenlo con **sus** decisiones y cifras.

### Hallazgos

- **El conjunto docente no sirve como conjunto de proyecto.** La evaluación de la sección
  5.2 lo muestra sin ambigüedad: faltan variable de fecha, variable discreta y una de alta
  cardinalidad. Descubrirlo aquí, y no en la Fase 2, es precisamente para lo que sirve
  evaluar el conjunto antes de comprometerse con él.

- **El rol analítico, y no el tipo de dato, es lo que define el trabajo pendiente.** En el
  diccionario hay cuatro variables `int64` que son cosas distintas: un identificador, tres
  binarias y una objetivo. El preprocesamiento de la Fase 2 se deriva de esa declaración.

- **La reproducibilidad es verificable o no existe.** El `requirements.txt` de este
  cuaderno se generó a partir de las versiones realmente presentes en el entorno, no de
  una lista escrita de memoria.

### Dificultades y cómo se resolvieron

- **Rutas.** Se usaron rutas relativas con `pathlib` y se acordó una única convención de
  carpetas de datos, compartida con el cuaderno de la Fase 2.

- **Dependencia del archivo de datos.** El cuaderno no falla si el conjunto todavía no
  está descargado: informa qué falta y continúa, porque la Fase 1 no depende de él.

### Decisiones que quedan abiertas

- Cuál será el conjunto definitivo del proyecto, una vez que supere los criterios.
- Si el conjunto se versiona en el repositorio o solo se documenta su descarga.
- Si el módulo `src/utilidades.py` crecerá o si parte de él pasará a clases en la Fase 3.

> **Declarar lo que queda abierto evidencia más dominio que presentar todo resuelto.** En
> esta fase no debería estarlo.

---

## 11. Verificación antes de entregar

**Cuaderno**

- Corre completo tras *Kernel → Restart Kernel and Run All Cells*, sin errores.
- La numeración de ejecución es continua.
- Cada bloque de código está precedido por una celda narrativa que explica su lógica.
- Todas las rutas son relativas y la semilla está declarada.

**Contenido técnico**

- La problemática, los objetivos, el alcance y las limitaciones están declarados.
- El entorno se verifica con evidencia (intérprete, entorno virtual, versiones).
- La procedencia del conjunto está documentada, con enlace y cita en APA 7.
- El diccionario declara el rol analítico de cada variable.
- El conjunto candidato fue evaluado contra los criterios del curso.
- Hay pruebas de casos normales, límite y excepciones.

**Archivos**

- `.gitignore` y `requirements.txt` en la raíz, ambos con contenido.
- `README.md` técnico, generado desde la configuración del proyecto.
- `docs/` con diccionario, evaluación del conjunto, vinculación y metadatos.
- `src/utilidades.py` importable.

**Repositorio**

- Carpeta F1 con este cuaderno, *commits* de todos los integrantes y `README` actualizado.
- El correo de Git coincide con el de la cuenta de GitHub.

---

*Cuaderno de la Fase 1 · MCDI500 · Magíster en Ciencia de Datos e Inteligencia
Artificial · Universidad Andrés Bello*